# Transition-case prevalence analysis


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import (
    hyperedge_homogeneity_list,
    hypergraph_disjointness,
    load_txt,
    theoretical_case_values,
    empirical_availability_values,
)

In [ ]:
# Dataset list (the two datasets used above).
dataset_names = ["contact-primary", "contact-high", "email-Enron-full", "email-Eu-full"]

datasets = {}
for i, name in enumerate(dataset_names):
    data_path = Path(f"../www21-hyperlap-master/dataset/{name}_graph.txt")
    H = load_txt(data_path)
    empirical = empirical_availability_values(H, seed=42 + i)
    theoretical = theoretical_case_values(H)
    homogeneity = hyperedge_homogeneity_list(H)
    disjointness = hypergraph_disjointness(H)

    datasets[name] = {
        "H": H,
        "empirical": empirical,
        "theoretical": theoretical,
        "homogeneity_mean": float(np.mean(homogeneity)),
        "homogeneity": homogeneity,
        "disjointness": disjointness,
    }

In [ ]:
case_labels = ["same", "middle", "other"]
emp_colors = ["#9bd9cf", "#f5b39d", "#b2c0d9"]
theo_colors = ["#2a9d8f", "#e76f51", "#5778a4"]

fig, axes = plt.subplots(2, 2, figsize=(8, 6), sharey=True)
axes = axes.flatten()

bar_w = 0.34
x = np.arange(len(case_labels), dtype=float)

for ax, name in zip(axes, dataset_names):
    values = datasets[name]
    empirical = np.asarray(values["empirical"], dtype=float)
    theoretical = np.asarray(values["theoretical"], dtype=float)
    dj = values["disjointness"] * 100
    hom = values["homogeneity_mean"]

    for i, _case in enumerate(case_labels):
        ax.bar(x[i] - bar_w / 2, empirical[i], width=bar_w, color=emp_colors[i])
        ax.bar(x[i] + bar_w / 2, theoretical[i], width=bar_w, color=theo_colors[i])

    ax.set_xticks(x)
    ax.set_xticklabels(case_labels, fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.set_title(name, fontsize=11)
    ax.grid(axis="y", alpha=0.25)

# Keep x-axis labels only on bottom row.
for ax in axes[:2]:
    ax.set_xlabel("")
    ax.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
for ax in axes[2:]:
    ax.set_xlabel("Case", fontsize=10)
for ax in axes[::2]:
    ax.set_ylabel("Fraction", fontsize=10)
for ax in axes[1::2]:
    ax.tick_params(axis="y", which="both", left=False, labelleft=False)

plt.tight_layout(rect=[0, 0, 1, 1])

out_path = "results/images/real_transitions.png"
fig.savefig(out_path, dpi=300, bbox_inches="tight", pad_inches=0.25)
print(f"Saved: {out_path}")

plt.show()

## Hyperedge Prediction Results


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from hypergraphx import Hypergraph


results_dir = Path("results")
datasets = ["contact_primary", "contact-high", "email-Enron-full", "email-Eu-full"]

rows = []
id_cols = ["method", "config_id", "p", "q", "r", "gamma", "delta", "num_walks", "walk_length", "dim"]

for dataset in datasets:
    dataset_dir = results_dir / dataset

    # Start from the wide file and map to its corresponding non-wide table.
    wide_path = next(dataset_dir.glob("*_wide.csv"))
    table_path = dataset_dir / wide_path.name.replace("_wide", "")
    df = pd.read_csv(table_path)
    df = df[(df["method"] == "hyper2vec") | ((df["r"] == 1.0) & (df["delta"] == 0.0))]

    per_config = (
        df.groupby(id_cols, dropna=False, as_index=False)
        .agg(
            auc=("auc", "mean"),
            ap=("ap", "mean"),
            auc_std=("auc", "std"),
            ap_std=("ap", "std"),
        )
        .fillna({"auc_std": 0.0, "ap_std": 0.0})
    )

    for method in ["harw", "hyper2vec"]:
        d = per_config[per_config["method"] == method]
        if d.empty:
            continue

        best = d.loc[d["auc"].idxmax()]

        rows.append(
            {
                "dataset": dataset,
                "method": method,
                "summary": "max",
                "auc": float(best["auc"]),
                "ap": float(best["ap"]),
                "auc_std": float(best["auc_std"]),
                "ap_std": float(best["ap_std"]),
                "config_id": best.get("config_id", None),
            }
        )

        rows.append(
            {
                "dataset": dataset,
                "method": method,
                "summary": "mean",
                "auc": float(d["auc"].mean()),
                "ap": float(d["ap"].mean()),
                "auc_std": float(d["auc_std"].mean()),
                "ap_std": float(d["ap_std"].mean()),
                "config_id": None,
            }
        )

summary_df = pd.DataFrame(rows).sort_values(["dataset", "method", "summary"]).reset_index(drop=True)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from hypergraphx import Hypergraph


results_dir = Path("results")
datasets = ["contact_primary", "contact-high", "email-Enron-full", "email-Eu-full"]

rows = []
id_cols = ["method", "config_id", "p", "q", "r", "gamma", "delta", "num_walks", "walk_length", "dim"]

for dataset in datasets:
    dataset_dir = results_dir / dataset

    wide_path = next(dataset_dir.glob("*_wide.csv"))
    table_path = dataset_dir / wide_path.name.replace("_wide", "")
    df = pd.read_csv(table_path)
    df = df[(df["method"] =="hyper2vec") | ((df["r"] == 1.0) & (df["delta"] == 0.0))]

    per_config = (
        df.groupby(id_cols, dropna=False, as_index=False)
        .agg(
            auc=("auc", "mean"),
            ap=("ap", "mean"),
            auc_std=("auc", "std"),
            ap_std=("ap", "std"),
        )
        .fillna({"auc_std": 0.0, "ap_std": 0.0})
    )

    for method in ["harw", "hyper2vec"]:
        d = per_config[per_config["method"] == method]
        if d.empty:
            continue

        best = d.loc[d["auc"].idxmax()]

        rows.append(
            {
                "dataset": dataset,
                "method": method,
                "summary": "max",
                "auc": float(best["auc"]),
                "ap": float(best["ap"]),
                "auc_std": float(best["auc_std"]),
                "ap_std": float(best["ap_std"]),
                "config_id": best.get("config_id", None),
            }
        )

        rows.append(
            {
                "dataset": dataset,
                "method": method,
                "summary": "mean",
                "auc": float(d["auc"].mean()),
                "ap": float(d["ap"].mean()),
                "auc_std": float(d["auc_std"].mean()),
                "ap_std": float(d["ap_std"].mean()),
                "config_id": None,
            }
        )

summary_df = pd.DataFrame(rows).sort_values(["dataset", "method", "summary"]).reset_index(drop=True)

In [ ]:
# Best (max) comparison per dataset: HARW vs Hyper2Vec
best = summary_df[summary_df["summary"] == "max"].copy()

rows = []
for dataset in sorted(best["dataset"].unique()):
    d = best[best["dataset"] == dataset]

    harw = d[d["method"] == "harw"].iloc[0]
    h2v = d[d["method"] == "hyper2vec"].iloc[0]

    auc_diff = float(harw["auc"] - h2v["auc"])
    ap_diff = float(harw["ap"] - h2v["ap"])

    auc_pct = float((auc_diff / h2v["auc"]) * 100) if h2v["auc"] != 0 else np.nan
    ap_pct = float((ap_diff / h2v["ap"]) * 100) if h2v["ap"] != 0 else np.nan

    rows.append(
        {
            "dataset": dataset,
            "harw_auc_max": float(harw["auc"]),
            "harw_ap_max": float(harw["ap"]),
            "harw_auc_std": float(harw["auc_std"]),
            "harw_config_id": harw.get("config_id", None),
            "hyper2vec_auc_max": float(h2v["auc"]),
            "hyper2vec_ap_max": float(h2v["ap"]),
            "hyper2vec_auc_std": float(h2v["auc_std"]),
            "hyper2vec_config_id": h2v.get("config_id", None),
            "auc_diff_harw_minus_hyper2vec": auc_diff,
            "ap_diff_harw_minus_hyper2vec": ap_diff,
            "auc_pct_change_h2v_to_harw": auc_pct,
            "ap_pct_change_h2v_to_harw": ap_pct,
        }
    )

best_compare_df = pd.DataFrame(rows).sort_values("dataset").reset_index(drop=True)

In [ ]:
best_compare_df[["dataset", "harw_auc_max", "hyper2vec_auc_max", "harw_auc_std", "hyper2vec_auc_std", "auc_diff_harw_minus_hyper2vec", "auc_pct_change_h2v_to_harw"]].round(4)

In [ ]:
from scipy.stats import wilcoxon

rows_best_folds = []
rows_tests = []

for dataset in sorted(best["dataset"].unique()):
    dataset_dir = results_dir / dataset
    wide_path = next(dataset_dir.glob("*_wide.csv"))
    table_path = dataset_dir / wide_path.name.replace("_wide", "")
    df_full = pd.read_csv(table_path)

    d = best[best["dataset"] == dataset]
    harw_cfg = d[d["method"] == "harw"].iloc[0]["config_id"]
    h2v_cfg = d[d["method"] == "hyper2vec"].iloc[0]["config_id"]

    harw_rows = df_full[(df_full["method"] == "harw") & (df_full["config_id"] == harw_cfg)].copy()
    h2v_rows = df_full[(df_full["method"] == "hyper2vec") & (df_full["config_id"] == h2v_cfg)].copy()

    harw_rows["dataset"] = dataset
    h2v_rows["dataset"] = dataset
    rows_best_folds.append(harw_rows)
    rows_best_folds.append(h2v_rows)

    paired = harw_rows[["fold", "auc", "ap"]].merge(
        h2v_rows[["fold", "auc", "ap"]],
        on="fold",
        how="inner",
        suffixes=("_harw", "_hyper2vec"),
    ).sort_values("fold")

    auc_diff = paired["auc_harw"] - paired["auc_hyper2vec"]
    ap_diff = paired["ap_harw"] - paired["ap_hyper2vec"]

    auc_test = wilcoxon(auc_diff)
    auc_stat, auc_p = float(auc_test.statistic), float(auc_test.pvalue)
    ap_test = wilcoxon(ap_diff)
    ap_stat, ap_p = float(ap_test.statistic), float(ap_test.pvalue)

    rows_tests.append(
        {
            "dataset": dataset,
            "harw_config_id": harw_cfg,
            "hyper2vec_config_id": h2v_cfg,
            "n_folds": int(len(paired)),
            "auc_wilcoxon_stat": auc_stat,
            "auc_wilcoxon_p": auc_p,
            "auc_significant_p_lt_0_05": bool(auc_p < 0.05) if pd.notna(auc_p) else False,
            "ap_wilcoxon_stat": ap_stat,
            "ap_wilcoxon_p": ap_p,
            "ap_significant_p_lt_0_05": bool(ap_p < 0.05) if pd.notna(ap_p) else False,
        }
    )

best_fold_rows_df = pd.concat(rows_best_folds, ignore_index=True)
wilcoxon_df = pd.DataFrame(rows_tests).sort_values("dataset").reset_index(drop=True)

In [ ]:
wilcoxon_df[["dataset", "auc_wilcoxon_p", "auc_significant_p_lt_0_05"]]